# agri-drone — Notebook 1: full experimental matrix

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashut0sh-mishra/agri-drone/blob/research-upgrade/notebooks/colab/01_run_matrix.ipynb)

Runs the research-upgrade evaluation matrix end-to-end on Colab.
- **Quick mode** (default, T4, ~2 h) = 8 cells
- **Full mode** (A100, ~12 h) = 2400 cells

Outputs land under `MyDrive/agri-drone/results_v2/`.


## 1. Mount Drive + clone repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!test -d agri-drone || git clone https://github.com/Ashut0sh-mishra/agri-drone.git
%cd /content/agri-drone
!git fetch --all && git checkout research-upgrade && git pull
!pip install -q -r requirements.txt


## 2. GPU sanity check


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU.'
print('CUDA', torch.version.cuda, 'device:', torch.cuda.get_device_name(0))


## 3. Data source

Pre-create in your Drive and upload datasets:
```
MyDrive/agri-drone/data/plantvillage/
MyDrive/agri-drone/data/PDT_datasets/
```
See `docs/data_availability.md` for URLs and licences.


In [ ]:
#@title Data source
DATA_SOURCE = 'gdrive' #@param ['gdrive','kaggle','zenodo']
DRIVE_DATA = '/content/drive/MyDrive/agri-drone/data'
DRIVE_OUT  = '/content/drive/MyDrive/agri-drone/results_v2'
import os, pathlib
pathlib.Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
if DATA_SOURCE == 'gdrive':
    assert os.path.isdir(DRIVE_DATA), f'Missing {DRIVE_DATA}. Upload dataset first.'
    !mkdir -p datasets/externals && ln -sfn "$DRIVE_DATA" datasets/externals/drive
elif DATA_SOURCE == 'kaggle':
    print('Configure ~/.kaggle/kaggle.json and pull per scripts/download_data.py')
elif DATA_SOURCE == 'zenodo':
    print('Replace with your Zenodo archive URL and unzip to datasets/externals/')
print('(SHA256 validation advisory per docs/data_availability.md)')


## 4. Mode selector


In [ ]:
#@title Mode
FULL_MODE = False #@param {type:'boolean'}
MATRIX_CONFIG = 'configs/matrix/full.yaml' if FULL_MODE else 'configs/matrix/quick.yaml'
if FULL_MODE:
    print('*** FULL MODE: 2400 cells, ~12 h on A100. Need Colab Pro. ***')
else:
    print('QUICK MODE: 8 cells, ~2 h on T4.')
print('config =', MATRIX_CONFIG)


## 5. Run the matrix


In [ ]:
!python evaluate/matrix/run_matrix.py --config $MATRIX_CONFIG \
    --out-dir $DRIVE_OUT/matrix --tracker none


## 6. Post-matrix analysis (stats v2, EML, baseline audit, PDT sweep)


In [ ]:
!python evaluate/statistical_tests.py --v2 --n-boot 10000
!python evaluate/eml_sensitivity.py
!python evaluate/matrix/audit_baseline.py
!python evaluate/pdt_v2.py --variant threshold_sweep


## 7. Auto-generate RESULTS_SUMMARY.md


In [ ]:
import json, pathlib
out = pathlib.Path(DRIVE_OUT); out.mkdir(parents=True, exist_ok=True)
def _load(rel):
    p = pathlib.Path('evaluate/results/v2') / rel
    return json.loads(p.read_text()) if p.exists() else {}
eml = _load('eml/headline_v4.json')
pdt = _load('pdt/threshold_sweep.json')
lines = [
    '# Results summary (auto-generated by notebook 1)',
    f'- Matrix config: `{MATRIX_CONFIG}`',
    f'- EML headline (INR/ha): {eml.get("headline_eml_inr_per_ha","[TO BE RE-RUN]")}',
    f'- EML sensitivity scenario: {eml.get("sensitivity_scenario_eml_inr_per_ha","[TO BE RE-RUN]")}',
    f'- PDT ROC-AUC: {pdt.get("roc_auc","[TO BE RE-RUN]")}',
    f'- PDT spec@90%sens: {pdt.get("specificity_at_90_sensitivity","[TO BE RE-RUN]")}',
]
(out / 'RESULTS_SUMMARY.md').write_text('\n'.join(lines))
print((out / 'RESULTS_SUMMARY.md').read_text())


## 8. Zip artifacts + print PR-comment command


In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M%S')
archive = f'/content/drive/MyDrive/agri-drone/results_v2_{stamp}'
shutil.make_archive(archive, 'zip', DRIVE_OUT)
print('Saved:', archive + '.zip')
print('\nPost summary back to PR locally with:')
print('  gh pr comment research-upgrade --body-file RESULTS_SUMMARY.md')
